# 01 · Bronze Ingest — Contoso Retail 360

Loads the raw landing CSVs into **Bronze** Delta tables, exactly as-is, with ingest metadata.
This notebook substitutes for the Data Factory Copy pipeline in the quick-test path.

**Prereqs:** attach `lh_retail` as the default Lakehouse, and upload the CSVs to
`Files/landing/` (products.csv, stores.csv, customers.csv, sales.csv).


In [ ]:
from pyspark.sql import functions as F

LANDING = "Files/landing"          # Lakehouse Files path
SCHEMA  = "bronze"                  # target schema (create it in the Lakehouse first)

sources = {
    "sales_raw":    f"{LANDING}/sales.csv",
    "products_raw": f"{LANDING}/products.csv",
    "stores_raw":   f"{LANDING}/stores.csv",
    "customers_raw":f"{LANDING}/customers.csv",
}

In [ ]:
for table, path in sources.items():
    df = (spark.read
          .option("header", True)
          .option("inferSchema", True)
          .csv(path))
    df = (df
          .withColumn("_ingest_ts", F.current_timestamp())
          .withColumn("_source_file", F.lit(path)))
    (df.write.format("delta").mode("overwrite")
       .saveAsTable(f"{SCHEMA}.{table}"))
    print(f"loaded {SCHEMA}.{table}: {df.count()} rows")

In [ ]:
# quick peek
display(spark.read.table(f"{SCHEMA}.sales_raw").limit(5))